In [1]:
import os, sys, json, pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

# ====== 0) 路径配置（按你 LogGenerator 的风格） ======
TEMP_DATA_STORAGE = "../../temp_data/aiops22"
RAW_DATA_ROOT     = f"{TEMP_DATA_STORAGE}/raw_data"

ANALYSIS_TRACE_DIR = f"{TEMP_DATA_STORAGE}/analysis/trace"
DATASET_TRACE_DIR  = f"{TEMP_DATA_STORAGE}/dataset/trace"

os.makedirs(ANALYSIS_TRACE_DIR, exist_ok=True)
os.makedirs(DATASET_TRACE_DIR, exist_ok=True)

print("RAW_DATA_ROOT      =", RAW_DATA_ROOT)
print("ANALYSIS_TRACE_DIR =", ANALYSIS_TRACE_DIR)
print("DATASET_TRACE_DIR  =", DATASET_TRACE_DIR)

# ====== 1) AIOps 基础设定（与你前面一致） ======
service_order_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

pod_list = [
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

node_list = ["node-1","node-2","node-3","node-4","node-5","node-6"]

def rename_service2pod(service: str):
    return [f"{service}-0", f"{service}-1", f"{service}-2", f"{service}2-0"]

# ====== 2) 枚举 raw_trace case（仿你 list_cases） ======
def list_trace_cases(raw_data_root: str):
    """
    返回 [(dataset_type, date, cloudbed, raw_trace_dir), ...]
    其中 raw_trace_dir = .../<dataset_type>/<date>/<cloudbed>/raw_trace
    """
    cases = []
    for dataset_type in sorted(os.listdir(raw_data_root)):
        dt_path = os.path.join(raw_data_root, dataset_type)
        if not os.path.isdir(dt_path):
            continue
        for date in sorted(os.listdir(dt_path)):
            date_path = os.path.join(dt_path, date)
            if not os.path.isdir(date_path):
                continue
            for cloudbed in sorted(os.listdir(date_path)):
                cb_path = os.path.join(date_path, cloudbed)
                raw_trace_dir = os.path.join(cb_path, "raw_trace")
                if os.path.isdir(raw_trace_dir) and os.path.exists(os.path.join(raw_trace_dir, "span_features.csv")):
                    cases.append((dataset_type, date, cloudbed, raw_trace_dir))
    return cases

trace_cases = list_trace_cases(RAW_DATA_ROOT)
trace_cases[:3], len(trace_cases)

def load_span_features(raw_trace_dir: str) -> pd.DataFrame:
    f = os.path.join(raw_trace_dir, "span_features.csv")
    if not os.path.exists(f):
        raise FileNotFoundError(f)
    return pd.read_csv(f)

# ====== 3) 特征名归一化（完全照你贴的 TraceGenerator.extract_entity_feature_name） ======
def extract_entity_feature_name_trace(feature_name: str) -> str:
    """
    feature_name 形如：
      "<intensity>; cmdb_id: adservice-1; type: upstream"
    源代码逻辑：
      cmdb_id = feature_name.split(';')[1].replace(...)
      return f'{feature_name.split(";")[0]};{cmdb_id};{feature_name.split(";")[2]}'
    """
    cmdb_id = feature_name.split(';')[1].replace('2-0', '').replace('-0', '').replace('-1', '').replace('-2', '')
    return f'{feature_name.split(";")[0]};{cmdb_id};{feature_name.split(";")[2]}'

# ====== 4) 计算 trace statistic（训练集） ======
def calculate_trace_statistic(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
):
    statistic_dict = {}
    data_dict = {}

    for dataset_type, date, cloudbed, raw_trace_dir in tqdm(cases, desc="scan trace cases"):
        if dataset_type == "test":
            continue
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        feature_df = load_span_features(raw_trace_dir)
        for feature_name in feature_df.columns:
            if feature_name == "timestamp":
                continue
            # 聚合了同一微服务的同一特征成一个列表，比如所有的<duration>;  cmdb_id: service; type: upstream
            exact_feature_name = extract_entity_feature_name_trace(feature_name)
            data_dict.setdefault(exact_feature_name, [])
            data_dict[exact_feature_name].extend(feature_df[feature_name].tolist())

    # 同一个service某一特征的统计值
    for feature_name, trace_data in data_dict.items():
        trace_data = np.array(trace_data, dtype=float)

        median = np.nanmedian(trace_data)
        percentile_1 = np.nanpercentile(trace_data, 1)
        percentile_99 = np.nanpercentile(trace_data, 99)
        q1 = np.nanpercentile(trace_data, 25)
        q3 = np.nanpercentile(trace_data, 75)
        mean = np.nanmean(trace_data)
        std = np.nanstd(trace_data)
        valid_ratio = float(np.count_nonzero(~np.isnan(trace_data)) / len(trace_data))

        statistic_dict[feature_name] = {
            "mean": float(mean),
            "std": float(std),
            "percentile_1": float(percentile_1),
            "q1": float(q1),
            "median": float(median),
            "q3": float(q3),
            "percentile_99": float(percentile_99),
            "valid_ratio": valid_ratio,
        }

    out_path = os.path.join(ANALYSIS_TRACE_DIR, "statistic.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(statistic_dict, f, indent=2)
    print("[OK] saved:", out_path)
    return statistic_dict

trace_stat = calculate_trace_statistic(trace_cases)
len(trace_stat)

# ====== 5) IQR 归一化（写 all_trace_features.pkl） ======
def z_score_trace_data_iqr(
    cases,
    skip_bad_case=True,
    bad_case_date="2022-03-24",
    bad_case_cloudbeds=("cloudbed-1", "cloudbed-2"),
):
    stat_path = os.path.join(ANALYSIS_TRACE_DIR, "statistic.json")
    with open(stat_path, "r", encoding="utf-8") as f:
        statistic_dict = json.load(f)

    out = {}
    for dataset_type, date, cloudbed, raw_trace_dir in tqdm(cases, desc="iqr-scale trace"):
        if skip_bad_case and date == bad_case_date and cloudbed in bad_case_cloudbeds:
            continue

        df = load_span_features(raw_trace_dir)

        # 拷贝并原地更新（与源代码“逐行写回”效果一致）
        df2 = df.copy()

        for col in df.columns:
            if col == "timestamp":
                continue
                
            exact_feature_name = extract_entity_feature_name_trace(col)
            if exact_feature_name not in statistic_dict:
                continue

            # 鲁棒性归一化，对某一pod的某一特征，进行鲁棒性归一化
            q1 = statistic_dict[exact_feature_name]["q1"]
            q3 = statistic_dict[exact_feature_name]["q3"]
            median = statistic_dict[exact_feature_name]["median"]
            iqr = q3 - q1

            if iqr != 0:
                df2[col] = (df2[col] - median) / iqr
            # iqr==0：源代码不更新，保持原值

        out.setdefault(dataset_type, {}).setdefault(date, {})[cloudbed] = {"span_features": df2}



    '''
        {
          dataset_type: {
            date: {
              cloudbed: {
                "span_features": DataFrame
              }
            }
          }
        }
        shape = (1440, 1 + 6 * pod数量)

    '''
    out_path = os.path.join(DATASET_TRACE_DIR, "all_trace_features.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(out, f)
    print("[OK] saved:", out_path)
    return out

all_trace = z_score_trace_data_iqr(trace_cases)

# ====== 6) time_interval：统一从 label pkl 读取（与 log/metric/trace/edge/y 对齐） ======
TIME_INTERVAL_PKL_DIR = os.path.join(TEMP_DATA_STORAGE, "dataset", "time_interval_and_label")

def get_time_interval_list(window_size, data_type):
    """
    data_type: 'train_valid' or 'test'
    return: list of [date, cloudbed, st, et]
    """
    fp = os.path.join(TIME_INTERVAL_PKL_DIR, f"time_interval_window_size_{window_size}.pkl")
    with open(fp, "rb") as f:
        obj = pickle.load(f)
    return obj["time_interval"][data_type]

def get_time_interval_trace_data(st, et, df):
    st, et = int(st), int(et)
    return np.array(df.query(f"{st} <= timestamp < {et}").iloc[:, df.columns != "timestamp"].values)

window_size_list = [5, 10, 15]  # 你按实验设置改

def build_case_index(cases):
    case_map = {}
    for dataset_type, date, cloudbed, raw_trace_dir in cases:
        case_map.setdefault(dataset_type, {}).setdefault(date, {})[cloudbed] = raw_trace_dir
    return case_map

trace_case_map = build_case_index(trace_cases)



def get_time_interval_trace_data(st, et, df):
    return np.array(df.query(f"{st} <= timestamp < {et}").iloc[:, df.columns != "timestamp"].values)

# ====== 7) generate_trace_data（完全对齐源 TraceGenerator.generate_trace_data） ======
def generate_trace_data():
    with open(os.path.join(DATASET_TRACE_DIR, "all_trace_features.pkl"), "rb") as f:
        all_trace_dict = pickle.load(f)

    # 源代码会把层级“拍平”为：all_trace_dict[date] = cloud_bed_data
    flat = {}
    for date_cloud_bed_data in all_trace_dict.values():
        for date, cloud_bed_data in date_cloud_bed_data.items():
            flat[date] = cloud_bed_data

    for window_size in tqdm(window_size_list, desc="window_size"):
        trace_dict = {}

        entity_features = []
        trace_name_list = []
        record_features = True

        for node in node_list:
            entity_features.append((node, (0, 0)))
        for svc in service_order_list:
            entity_features.append((svc, (0, 0)))

        for data_type in ["train_valid", "test"]:
            time_interval_label_list = get_time_interval_list(window_size, data_type)
            trace_dict[data_type] = []

            for time_interval in tqdm(time_interval_label_list, desc=f"{data_type} intervals", leave=False):
                date, cloudbed, st, et = time_interval[0], time_interval[1], int(time_interval[2]), int(time_interval[3])

                trace_df = flat[date][cloudbed]["span_features"]
                temp = get_time_interval_trace_data(st, et, trace_df)
                temp_name_list = list(trace_df.columns[trace_df.columns != "timestamp"])

                data = []
                feature_index = 0

                for svc in service_order_list:
                    pods = rename_service2pod(svc)
                    for pod in pods:
                        pod_related_trace_name_list = []
                        for position_type in ["upstream", "current", "downstream"]:
                            for feature_type in ["<intensity>", "<duration>"]:
                                feature_name = f"{feature_type}; cmdb_id: {pod}; type: {position_type}"
                                if feature_name not in temp_name_list:
                                    continue
                                data.append(temp[:, temp_name_list.index(feature_name)])
                                pod_related_trace_name_list.append(feature_name)

                        if record_features:
                            entity_features.append((pod, (feature_index, feature_index + len(pod_related_trace_name_list))))
                            feature_index += len(pod_related_trace_name_list)
                            trace_name_list.extend(pod_related_trace_name_list)

                trace_dict[data_type].append(np.array(data).transpose())
                record_features = False

        out_path = os.path.join(DATASET_TRACE_DIR, f"trace_window_size_{window_size}.pkl")
        '''
            保持的数据结构
            {
                "trace_data": trace_dict,
                "entity_features": entity_features,
                "trace_names": trace_name_list,
            }
            ====================================================
            trace_dict = {
                "train_valid": [array1, array2, ...],
                "test":        [array1, array2, ...]
            }
            每个 array 形状为(T_window, F)，F为所有 pod 的 trace 特征数量
            ====================================================
            entity_features为每个实体（node / service / pod）对应特征矩阵的列索引区间
            [
              ("node-1", (0,0)),
              ...
              ("adservice", (0,0)),
              ...
              ("adservice-0", (0,6)),
              ("adservice-1", (6,12)),
              ...
            ]
            ====================================================
            trace_names为特征名
            [
              "<intensity>; cmdb_id: adservice-0; type: upstream",
              "<duration>; cmdb_id: adservice-0; type: upstream",
              ...
            ]
            XT∈Rt×6n

        '''
        with open(out_path, "wb") as f:
            pickle.dump({
                "trace_data": trace_dict,
                "entity_features": entity_features,
                "trace_names": trace_name_list,
            }, f)
        print("[OK] saved:", out_path)

generate_trace_data()

scan trace cases:  20%|██        | 3/15 [00:00<00:00, 29.43it/s]

RAW_DATA_ROOT      = ../../temp_data/aiops22/raw_data
ANALYSIS_TRACE_DIR = ../../temp_data/aiops22/analysis/trace
DATASET_TRACE_DIR  = ../../temp_data/aiops22/dataset/trace


iqr-scale trace:  13%|█▎        | 2/15 [00:00<00:00, 18.92it/s]

[OK] saved: ../../temp_data/aiops22/analysis/trace/statistic.json


train_valid intervals:   1%|          | 2/300 [00:00<00:19, 15.59it/s]

...

[OK] saved: ../../temp_data/aiops22/dataset/trace/trace_window_size_5.pkl



train_valid intervals: 100%|██████████| 300/300 [00:19<00:00, 14.87it/s]
                                                                        
train_valid intervals:   1%|          | 2/300 [00:00<00:19, 15.31it/s]

[OK] saved: ../../temp_data/aiops22/dataset/trace/trace_window_size_10.pkl



train_valid intervals: 100%|██████████| 300/300 [00:19<00:00, 15.18it/s]
                                                                        
window_size: 100%|██████████| 3/3 [01:48<00:00, 36.05s/it]       

[OK] saved: ../../temp_data/aiops22/dataset/trace/trace_window_size_15.pkl


In [1]:
import pickle, numpy as np

pkl_path = "../../temp_data/aiops22/dataset/trace/all_trace_features.pkl"
with open(pkl_path, "rb") as f:
    all_trace = pickle.load(f)

# 找到你刚才那个 case
df2 = all_trace["training_data_with_faults"]["2022-03-20"]["cloudbed-1"]["span_features"]

vals = df2.drop(columns=["timestamp"]).to_numpy()
print("after IQR: min/max/maxabs:", np.nanmin(vals), np.nanmax(vals), np.nanmax(np.abs(vals)))
print("after IQR: abs 99p:", np.nanpercentile(np.abs(vals), 99))

after IQR: min/max/maxabs: -6.520864981493164 9329.69199563628 9329.69199563628
after IQR: abs 99p: 6.520864981493164
